## Подключение к БД

In [5]:
import sys
from pathlib import Path

# добавление папки src в места поиска модулей
sys.path.append(str(Path.cwd().parent / 'src'))

from db import query

## Проверка подключения к базе и вывод названий всех таблиц в БД

In [6]:
query('SELECT name FROM sqlite_master WHERE type = "table"')

,name
0,users
1,devices
2,app_releases
3,screens
4,sessions
5,screen_views
6,events
7,orders
8,ab_assignments


# Просмотр всех таблиц и столбцов в них для планирования анализа данных


In [28]:
query('SELECT'
      ' sm.name AS table_name, '
      ' t_info.name AS column_name, '
      ' t_info.type AS column_type '
      'FROM sqlite_master sm '
      'JOIN pragma_table_info(sm.name) t_info '
      'WHERE sm.type = "table" '
      'ORDER BY sm.name, t_info.cid;')

,table_name,column_name,column_type
0,ab_assignments,user_id,INTEGER
1,ab_assignments,experiment,TEXT
2,ab_assignments,variant,TEXT
3,ab_assignments,assigned_at,TEXT
4,app_releases,platform,TEXT
5,app_releases,app_version,TEXT
6,app_releases,released_at,TEXT
7,app_releases,notes,TEXT
8,devices,device_id,TEXT
9,devices,user_id,INTEGER


# Анализ дат на достоверность

In [35]:
query('''
SELECT "sessions start" AS tbl, COUNT(*) AS rows,
MIN(started_at) AS min_date, MAX(started_at) AS max_date
FROM sessions
UNION ALL
SELECT "sessions end", COUNT(*),
MIN(ended_at), MAX(ended_at)
FROM sessions
UNION ALL
SELECT "orders create", COUNT(*),
MIN(created_at), MAX(created_at)
FROM orders
UNION ALL
SELECT "screen_views enter", COUNT(*),
MIN(entered_at), MAX(entered_at)
FROM screen_views
UNION ALL
SELECT "users registrate", COUNT(*),
MIN(registered_at), MAX(registered_at)
FROM users
UNION ALL
SELECT "devices first seen", COUNT(*),
MIN(first_seen_at), MAX(first_seen_at)
FROM devices
UNION ALL
SELECT "app_releases release", COUNT(*),
MIN(released_at), MAX(released_at)
FROM app_releases
UNION ALL
SELECT "ab_assignments assign", COUNT(*),
MIN(assigned_at), MAX(assigned_at)
FROM ab_assignments
UNION ALL
SELECT "events client", COUNT(*),
MIN(client_ts), MAX(client_ts)
FROM events
UNION ALL
SELECT "events server", COUNT(*),
MIN(server_ts), MAX(server_ts)
FROM events
UNION ALL
SELECT "screens info", COUNT(*),
NULL, NULL
FROM screens;
''')

,tbl,rows,min_date,max_date
0,sessions start,49191,2026-05-31 21:40:05,2026-08-31 18:48:39
1,sessions end,49191,2026-05-31 21:40:55,2026-08-31 18:56:54
2,orders create,21954,2026-05-31 23:20:55,2026-08-31 18:50:38
3,screen_views enter,286078,2026-05-31 21:40:05,2026-08-31 18:56:48
4,users registrate,9006,2025-01-01 07:08:27,2026-08-30 18:40:34
5,devices first seen,9006,2025-01-01 07:08:27,2026-08-30 18:40:34
6,app_releases release,9,2026-05-20,2026-08-13
7,ab_assignments assign,9031,2026-06-30 21:01:00,2026-08-31 18:36:19
8,events client,270273,2025-05-04 08:33:10,2027-07-11 03:44:33
9,events server,270273,2026-05-31 21:40:05,2026-08-31 23:39:16


## По данной таблице можно увидеть, что:
1. Самое позднее время во всех таблицах это 18:56 и оно почти везде одинаковое, значит можно сделать вывод, что в БД время отличается от UTC Узбекистана (+5 часов), если исправить, то время увеличится на 5 часов - это и будет граница периода по ТЗ
2. По объему данных:
    1)  ~ 9 000 пользователей - верно
    2)  ~ 49 000 сессий - верно
    3)  ~ 286 000 просмотров экранов - верно
    4)  ~ 270 000 событий - верно
    5)  ~ 22 000 заказов - верно
3. У таблицы  app_releases формат даты отличен от остальных. Требует внимания!
4. В таблицах users и devices полное совпадение минимумов и максимумов. То есть 1 пользователь = 1 устройство. Нужно потом разобраться почему так? и почему момент регистрации совпадает с первым подключением устройства до секунды?
5. events: Почему у события на стороне Клиента не совпадают с датами событий на стороне Сервера? Почему события сервера вписываются в диапазон по ТЗ, а события на стороне Клиента имеют больший разброс по датам (количество строк у них одинаковое) и максимум уходит в будущее? Может ли проблема быть на стороне Клиента?

Время на стороне Клиента ненадежно, поэтому для дальнейшего анализа буду использовать время Сервера. Со временем на стороне Клиента разберусь в следующих шагах
